# Conversational RAG with Chainlit: Document Ingestion

**Stack:** OpenRouter · Hugging Face BGE-M3 embeddings · Qdrant

Set `MODEL_TIER` to `"free"` (NVIDIA Nemotron) or explicitly to `"paid"` (Nex-N2-Mini). The paid tier requires OpenRouter credit and creates billable requests; there is no automatic fallback from free to paid. NVIDIA's free endpoint may log prompts, so use only public, non-sensitive documents.

Progressive journey:
- Ingest a single document and retrieve chunks
- Scale to multiple companies
- Explore metadata and add filters
- Build an agent — simple first, then filter-aware

## 1. Setup

In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

from ragwire import RAGWire, setup_logging
import ragwire

print(ragwire.__version__)


1.2.9


In [2]:
logger = setup_logging(log_level="INFO")

## 2. Ingest a Single Document and Retrieve

**Model:** selected OpenRouter tier · **Embedding:** `BAAI/bge-m3` · **Vector Store:** Qdrant (`finance-rag-qdrant`)

In [5]:
MODEL_TIER = "paid"  # change to "paid" explicitly when desired
rag = RAGWire("5config_openrouter_qdrant.yaml", model_tier=MODEL_TIER)
print(f"Using {rag.model_tier} model: {rag.config['llm']['model']}")

2026-09-01 02:29:59,245 - ragwire.core.pipeline - INFO - Loading configuration from 5config_openrouter_qdrant.yaml
2026-09-01 02:29:59,249 - ragwire.core.pipeline - INFO - Selected paid LLM model: nex-agi/nex-n2-mini
2026-09-01 02:29:59,260 - ragwire.core.pipeline - INFO - Document loader initialized
2026-09-01 02:29:59,261 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-09-01 02:30:06,788 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=huggingface)
2026-09-01 02:30:06,792 - ragwire.core.pipeline - INFO - Metadata extractor loaded from: finance_metadata.yaml
2026-09-01 02:30:06,793 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=openrouter, model=nex-agi/nex-n2-mini)
2026-09-01 02:30:06,806 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://localhost:6333 (timeout=300s)
2026-09-01 02:30:06,811 - ragwire.core.pipeline - INFO - Using existing collection: finance-rag-qdrant
2026-09-01 02:30:07,145 - ragwire.core.pipeline - INFO - Vector store initialized
2026-09-01 02:30:07,145 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-09-01 02:30:07,145 - ragwire.core.pipeline - INFO - RAG pipeline initialized successfully
Using paid model: nex-agi/nex-n2-mini


# RAGWire and Agentic RAG
- Upload set of documents
- Retrieval 
- Agentic RAG

## 3. Scale to Multiple Companies

Ingest all three 10-K filings at once. RAGWire deduplicates — re-running skips already-ingested files.

In [6]:
rag.ingest_directory('../data/finance_data')

2026-09-01 02:30:11,785 - ragwire.core.pipeline - INFO - Found 7 file(s) in ../data/finance_data
2026-09-01 02:30:11,785 - ragwire.core.pipeline - INFO - Starting ingestion of 7 documents


Ingesting:   0%|          | 0/7 [00:00<?, ?file/s]

2026-09-01 02:30:11,797 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/apple 10-k 2024.pdf
2026-09-01 02:30:11,804 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/amazon 10k 2025.pdf
2026-09-01 02:30:11,814 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/google 10-k 2024.pdf
2026-09-01 02:30:11,820 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/Apple_10k_2025.pdf
2026-09-01 02:30:11,828 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/GOOG-10-K-2025.pdf
2026-09-01 02:30:11,841 - ragwire.core.pipeline - INFO - Skipping (already ingested): ../data/finance_data/Facebook-10k-2025.pdf
2026-09-01 02:32:05,833 - ragwire.core.pipeline - INFO - Processed ../data/finance_data/amazon 10-k 2024.pdf: 43 chunks


Ingesting: 100%|██████████| 7/7 [01:54<00:00, 16.29s/file]

2026-09-01 02:32:05,906 - ragwire.core.pipeline - INFO - Ingestion complete: 1/7 documents


{'total': 7,
 'processed': 1,
 'skipped': 6,
 'failed': 0,
 'chunks_created': 43,
 'errors': []}

## 4. Explore Metadata

RAGWire extracts company name, doc type, and fiscal year during ingestion. Let's inspect what's stored.

In [7]:
rag.discover_metadata_fields()

['source',
 'file_name',
 'file_type',
 'file_hash',
 'chunk_id',
 'chunk_hash',
 'chunk_index',
 'total_chunks',
 'created_at',
 'company_name',
 'doc_type',
 'fiscal_year',
 'fiscal_quarter']

In [8]:
rag.filter_fields

['company_name', 'doc_type', 'fiscal_year', 'fiscal_quarter']

## 5. Manual Metadata Filters

The simple agent above can mix up companies when all three are in the same collection.
Filters let us pin retrieval to a specific company, year, or doc type.

In [9]:
query = "what is apple's revenue in 2025?"
results = rag.retrieve(query=query)

2026-09-01 02:38:58,484 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


In [10]:
results

[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_17', 'chunk_hash': '6067767cea96df0195721bf9386f28bcdfb3f1cf2d1cdbc5bf7959b3ed797d50', 'chunk_index': 17, 'total_chunks': 40, 'created_at': '2026-08-31T19:39:25.092173+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': 'b3c7ae1c-459c-4a64-ba09-d937c131ad35', '_collection_name': 'finance-rag-qdrant'}, page_content='Products and Services Performance\nThe following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions):\n|        | 2025       | Change | 2024     | Change | 2023     |\n| ------ | ---------- | ------ | -------- | ------ | -------- |\n| iPhone | $ 209,586  | 4 % $  | 201,183  | — % $  | 200,583  |\n| Mac    | 3

In [11]:
results = rag.retrieve(query=query, filters={'company_name':'apple inc.'})
results

2026-09-01 02:39:02,764 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': 'e519d5a46266050a9201e4f2cbbcdd22b5daca6d2c0046d414aa314f601cd4f8', 'chunk_index': 16, 'total_chunks': 40, 'created_at': '2026-08-31T19:39:25.092041+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': '7d3b603c-e93f-4115-b088-a1248d048488', '_collection_name': 'finance-rag-qdrant'}, page_content='Company Stock Performance\nThe following graph shows a comparison of five-year cumulative total shareholder return, calculated on a dividend-reinvested basis, for the Company, the S&P\n500 Index and the Dow Jones U.S. Technology Total Stock Market Index. The graph assumes $100 was invested in each of the Company’s common stock, the\nS&P

In [12]:
query = 'what is revenue on Google?'
results = rag.retrieve(query=query, filters={'company_name':'alphabet inc.'})
results

2026-09-01 02:39:05,016 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is revenue on Google?...


[Document(metadata={'source': '../data/finance_data/google 10-k 2024.pdf', 'file_name': 'google 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9', 'chunk_id': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9_26', 'chunk_hash': '0b768ab9e580b9b14a1819d6b285b8ca2658ef465dd6c15043494f7f0b29992a', 'chunk_index': 26, 'total_chunks': 57, 'created_at': '2026-08-31T19:47:15.042846+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2024, 'fiscal_quarter': None, '_id': 'c6399806-2d7d-4e98-8a15-6f4edf2a2abc', '_collection_name': 'finance-rag-qdrant'}, page_content='| Table of Contents |     | Alphabet Inc. |\n| ----------------- | --- | ------------- |\nGoogle subscriptions, platforms, and devices\nGoogle subscriptions, platforms, and devices revenues increased $5.7 billion from 2023 to 2024. The growth was primarily driven by an\nincrease in subscription revenues, largely from growth in th

In [13]:
query = 'what is revenue of Google in 2024?'
results = rag.retrieve(query=query, filters={'company_name':'alphabet inc.', 'fiscal_year': 2024})
results

2026-09-01 02:39:07,089 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is revenue of Google in 2024?...


[Document(metadata={'source': '../data/finance_data/google 10-k 2024.pdf', 'file_name': 'google 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9', 'chunk_id': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9_26', 'chunk_hash': '0b768ab9e580b9b14a1819d6b285b8ca2658ef465dd6c15043494f7f0b29992a', 'chunk_index': 26, 'total_chunks': 57, 'created_at': '2026-08-31T19:47:15.042846+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2024, 'fiscal_quarter': None, '_id': 'c6399806-2d7d-4e98-8a15-6f4edf2a2abc', '_collection_name': 'finance-rag-qdrant'}, page_content='| Table of Contents |     | Alphabet Inc. |\n| ----------------- | --- | ------------- |\nGoogle subscriptions, platforms, and devices\nGoogle subscriptions, platforms, and devices revenues increased $5.7 billion from 2023 to 2024. The growth was primarily driven by an\nincrease in subscription revenues, largely from growth in th

## 6. Auto-Filter

RAGWire can extract filters from the query automatically — no need to pass them manually.

In [14]:
rag._auto_filter

False

In [15]:
rag._auto_filter = True

In [16]:
query = "what is apple's revenue in 2025?"
results = rag.retrieve(query=query)
results

2026-09-01 02:39:18,379 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc.', 'fiscal_year': 2025}
2026-09-01 02:39:18,516 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is apple's revenue in 2025?...


[Document(metadata={'source': '../data/finance_data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_19', 'chunk_hash': 'c4772307f3b6c6d4802d8a1e11f9f0de51b667e4a66ffb4b7f7f29dd53e9be8f', 'chunk_index': 19, 'total_chunks': 40, 'created_at': '2026-08-31T19:39:25.092263+00:00', 'company_name': 'apple inc.', 'doc_type': '10-k', 'fiscal_year': 2025, 'fiscal_quarter': None, '_id': 'a2608aa4-5ddb-414b-b06d-51f90216b3ad', '_collection_name': 'finance-rag-qdrant'}, page_content='Item 8. Financial Statements and Supplementary Data\nIndex to Consolidated Financial Statements Page\nConsolidated Statements of Operations for the years ended September 27, 2025, September 28, 2024 and September 30, 2023 29\nConsolidated Statements of Comprehensive Income for the years ended September 27, 2025, September 28, 2024 and S

In [17]:
query = "what is google's revenue in 2024?"
results = rag.retrieve(query=query)
results

2026-09-01 02:39:23,842 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'alphabet inc.', 'fiscal_year': 2024}
2026-09-01 02:39:23,983 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: what is google's revenue in 2024?...


[Document(metadata={'source': '../data/finance_data/google 10-k 2024.pdf', 'file_name': 'google 10-k 2024.pdf', 'file_type': 'pdf', 'file_hash': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9', 'chunk_id': '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9_26', 'chunk_hash': '0b768ab9e580b9b14a1819d6b285b8ca2658ef465dd6c15043494f7f0b29992a', 'chunk_index': 26, 'total_chunks': 57, 'created_at': '2026-08-31T19:47:15.042846+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_year': 2024, 'fiscal_quarter': None, '_id': 'c6399806-2d7d-4e98-8a15-6f4edf2a2abc', '_collection_name': 'finance-rag-qdrant'}, page_content='| Table of Contents |     | Alphabet Inc. |\n| ----------------- | --- | ------------- |\nGoogle subscriptions, platforms, and devices\nGoogle subscriptions, platforms, and devices revenues increased $5.7 billion from 2023 to 2024. The growth was primarily driven by an\nincrease in subscription revenues, largely from growth in th

In [18]:
rag._auto_filter = False